<a href="https://colab.research.google.com/github/difanaya/PCVK_Ganjil_2026/blob/main/Modul3/Modul3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Modul 3 - Operasi Citra Sederhana


## D1. Operasi Citra Sederhana



In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt
import glob
import math
from google.colab.patches import cv2_imshow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# sesuaikan dengan folder di drive masing-masing
folder = '/content/drive/MyDrive/PCVK/Images/'
print(os.listdir(folder))

In [ ]:
def tampil(img1, img2, judul1='asli', judul2='hasil'):
    plt.figure(figsize=(11,4.5))
    plt.subplot(1,2,1)
    if img1.ndim == 2:
        plt.imshow(img1, cmap='gray', vmin=0, vmax=255)
    else:
        plt.imshow(cv.cvtColor(img1, cv.COLOR_BGR2RGB))
    plt.title(judul1)
    plt.subplot(1,2,2)
    if img2.ndim == 2:
        plt.imshow(img2, cmap='gray', vmin=0, vmax=255)
    else:
        plt.imshow(cv.cvtColor(img2, cv.COLOR_BGR2RGB))
    plt.title(judul2)
    plt.show()

### Transformasi Linier Brightness



In [ ]:
print(' Mengubah tingkat kecerahan citra ')
print('----------------------------------')
try:
    brightness = int(input('Masukkan nilai kecerahan: '))
except ValueError:
    print('Error, not a number')

original = cv.imread(folder + 'houses.jpg')
brightness_image = np.zeros(original.shape, original.dtype)

# akses per piksel
# np.clip digunakan untuk melakukan truncate pixel (nilai dibatasi antara 0-255)
for y in range(original.shape[0]):
    for x in range(original.shape[1]):
        for c in range(original.shape[2]):
            brightness_image[y,x,c] = np.clip(original[y,x,c] + brightness, 0, 255)

final_frame = cv.hconcat((original, brightness_image))
cv2_imshow(final_frame)

In [ ]:
# cara simple tanpa for loop, hasilnya sama tapi jauh lebih cepat
brightness_image2 = cv.convertScaleAbs(original, beta=brightness)
cv2_imshow(cv.hconcat((original, brightness_image2)))

In [ ]:
coba = np.array([[[10, 240, 250]]], np.uint8)
naik = np.clip(coba.astype(int) + 20, 0, 255)
turun = np.clip(naik - 20, 0, 255)
print('awal      :', coba.ravel())
print('tambah 20 :', naik.ravel())
print('kurang 20 :', turun.ravel())

# TUGAS PRAKTIKUM

## 1. Inverse Citra

Rumusnya g(x) = 255 - f(x). Nilai pixel dibalik terhadap nilai tengah, hasilnya citra negative.

In [ ]:
peppers = cv.imread(folder + 'peppers.jpg')
inverse = 255 - peppers

tampil(peppers, inverse, 'citra asli', 'inverse / negative')

In [ ]:
# kalau di-inverse dua kali harusnya balik ke citra awal
print(np.array_equal(255 - inverse, peppers))

## 2. Transformasi Contrast

Ada dua cara di ulasan teori. Pertama g(x,y) = a * f(x,y) + b, kedua pakai contrast correction
factor F = 259(C+255) / 255(259-C) terus R' = F(R-128) + 128. Saya coba dua-duanya.

In [ ]:
print(' Mengubah kontras dan tingkat kecerahan citra ')
print('----------------------------------------------')
try:
    kecerahan = int(input('Masukkan tingkat kecerahan : '))
    kontras = float(input('Masukkan kontras : '))
except ValueError:
    print('Error, not a number')

houses = cv.imread(folder + 'houses.jpg')

# cara 1: g(x,y) = a * f(x,y) + b
hasil1 = np.clip(houses.astype(float) * kontras + kecerahan, 0, 255).astype(np.uint8)
cv2_imshow(cv.hconcat((houses, hasil1)))

In [ ]:
# cara 2: pakai contrast correction factor
def contrast(img, C):
    F = (259.0 * (C + 255.0)) / (255.0 * (259.0 - C))   # harus double, bukan integer
    print('nilai F =', F)
    hasil = F * (img.astype(float) - 128) + 128
    return np.clip(hasil, 0, 255).astype(np.uint8)

hasil2 = contrast(houses, 60)
tampil(houses, hasil2, 'citra asli', 'contrast C = 60')

In [ ]:
# lihat pengaruh nilai C ke histogram
plt.figure(figsize=(14,6))
for i, C in enumerate([-80, 0, 60, 120]):
    h = contrast(houses, C)
    plt.subplot(2,4,i+1)
    plt.imshow(cv.cvtColor(h, cv.COLOR_BGR2RGB))
    plt.title('C = ' + str(C))
    plt.axis('off')
    plt.subplot(2,4,i+5)
    plt.hist(cv.cvtColor(h, cv.COLOR_BGR2GRAY).ravel(), bins=64, range=(0,255))
    plt.yticks([])
plt.show()

## 3. Transformasi Logarithmic Brightness

s = c * log(1 + r), c konstanta, r nilai grey level input, s nilai grey level output.

In [ ]:
print(' Mengubah tingkat kecerahan citra dengan Transformasi Log ')
print('----------------------------------------------------------')
try:
    c = float(input('Masukkan nilai kecerahan: '))
except ValueError:
    print('Error, not a number')

log_image = c * np.log(1 + houses.astype(float))
log_image = np.clip(log_image, 0, 255).astype(np.uint8)

cv2_imshow(cv.hconcat((houses, log_image)))

In [ ]:
# bandingkan kurva pemetaannya dengan linier brightness
r = np.arange(256)
plt.figure(figsize=(6,4))
plt.plot(r, r, '--', color='gray', label='tanpa transformasi')
plt.plot(r, np.clip(r + 30, 0, 255), label='linier b = 30')
plt.plot(r, np.clip(c * np.log(1 + r), 0, 255), label='log c = ' + str(c))
plt.xlabel('input')
plt.ylabel('output')
plt.legend()
plt.show()

## 4. Grayscale (Averaging, Lightness, Luminance)

- averaging = (R + G + B) / 3
- lightness = (max(R,G,B) + min(R,G,B)) / 2
- luminance = 0.21R + 0.72G + 0.07B

In [ ]:
b, g, r = cv.split(peppers.astype(float))

averaging = np.clip((r + g + b) / 3, 0, 255).astype(np.uint8)
lightness = np.clip((peppers.astype(float).max(axis=2) + peppers.astype(float).min(axis=2)) / 2, 0, 255).astype(np.uint8)
luminance = np.clip(0.21*r + 0.72*g + 0.07*b, 0, 255).astype(np.uint8)

tampil(peppers, averaging, 'citra asli', 'a. averaging')
tampil(peppers, lightness, 'citra asli', 'b. lightness')
tampil(peppers, luminance, 'citra asli', 'c. luminance')

In [ ]:
for nama, hasil in [('averaging', averaging), ('lightness', lightness), ('luminance', luminance)]:
    print(nama, '-> mean = %.2f , std = %.2f' % (hasil.mean(), hasil.std()))

plt.figure(figsize=(7,4))
plt.hist(averaging.ravel(), bins=64, range=(0,255), histtype='step', label='averaging')
plt.hist(lightness.ravel(), bins=64, range=(0,255), histtype='step', label='lightness')
plt.hist(luminance.ravel(), bins=64, range=(0,255), histtype='step', label='luminance')
plt.legend()
plt.show()

## 5. Menampilkan Warna Tertentu, Sisanya Grayscale



In [ ]:
hsv = cv.cvtColor(peppers, cv.COLOR_BGR2HSV)

mask1 = cv.inRange(hsv, np.array([0,90,60]), np.array([10,255,255]))
mask2 = cv.inRange(hsv, np.array([170,90,60]), np.array([180,255,255]))
mask = cv.bitwise_or(mask1, mask2)

# dibersihin dikit biar tidak ada bintik-bintik
mask = cv.morphologyEx(mask, cv.MORPH_OPEN, np.ones((5,5), np.uint8))
mask = cv.morphologyEx(mask, cv.MORPH_CLOSE, np.ones((9,9), np.uint8))

abu = cv.cvtColor(luminance, cv.COLOR_GRAY2BGR)
hasil_merah = np.where(mask[:,:,None] > 0, peppers, abu)

plt.figure(figsize=(14,4.5))
plt.subplot(1,3,1); plt.imshow(cv.cvtColor(peppers, cv.COLOR_BGR2RGB)); plt.title('citra asli')
plt.subplot(1,3,2); plt.imshow(mask, cmap='gray'); plt.title('mask warna merah')
plt.subplot(1,3,3); plt.imshow(cv.cvtColor(hasil_merah, cv.COLOR_BGR2RGB)); plt.title('merah tetap, sisanya grayscale')
plt.show()

## 6. Gamma Correction

Rumus gamma: I' = 255 * (I/255)^gamma

Rumus gamma correction pakai invers gamma: I' = 255 * (I/255)^(1/gamma)

In [ ]:
print(' Gamma Correction pada citra ')
print('------------------------------')
try:
    gamma = float(input('Masukkan nilai Gamma: '))
except ValueError:
    print('Error, not a number')

def gamma_correction(img, gamma):
    # pakai LUT biar tidak looping per pixel
    lut = np.array([np.clip(((i/255.0) ** (1.0/gamma)) * 255.0, 0, 255) for i in range(256)], np.uint8)
    return cv.LUT(img, lut)

hasil_gamma = gamma_correction(houses, gamma)
tampil(houses, hasil_gamma, 'Citra Asli', 'Gamma Correction (gamma = ' + str(gamma) + ')')

In [ ]:
plt.figure(figsize=(16,3.5))
for i, gm in enumerate([0.5, 1.0, 2.0, 3.0, 6.0]):
    plt.subplot(1,5,i+1)
    plt.imshow(cv.cvtColor(gamma_correction(houses, gm), cv.COLOR_BGR2RGB))
    plt.title('gamma = ' + str(gm))
    plt.axis('off')
plt.show()

In [ ]:
# kurva gamma vs gamma correction
i = np.arange(256)
plt.figure(figsize=(11,4))
plt.subplot(1,2,1)
for gm in [0.25, 0.5, 1.0, 1.5, 2.0]:
    plt.plot(i, 255 * (i/255.0)**gm, label='gamma = ' + str(gm))
plt.title('Gamma Curves'); plt.legend(fontsize=8)
plt.subplot(1,2,2)
plt.plot(i, i, label='gamma = 1')
plt.plot(i, 255*(i/255.0)**2.0, label='gamma = 2')
plt.plot(i, 255*(i/255.0)**(1/2.0), label='gamma correction = 2')
plt.title('Gamma vs Gamma Correction'); plt.legend(fontsize=8)
plt.show()

## 7. Simulasi Image Depth

level = 255 / (2^bit_depth - 1), terus C' = round(C / level) * level.

Ini cuma simulasi, bitnya tidak benar-benar berkurang, cuma variasi warnanya saja yang dibatasi.

In [ ]:
try:
    bit_depth = int(input('Masukkan bit depth tujuan (1-8): '))
except ValueError:
    print('Error, not a number')

level = 255 / (pow(2, bit_depth) - 1)
original = cv.imread(folder + 'peppers.jpg', cv.IMREAD_GRAYSCALE)
depth_image = np.zeros(original.shape, original.dtype)

depth_image = (np.round(original.astype(float) / level) * level).astype(np.uint8)

print('Bit depth awal   : 8 bit')
print('Bit depth tujuan :', bit_depth, 'bit')
print('Jumlah level     :', pow(2, bit_depth))
print('Nilai level      : %.4f' % level)
print('Warna unik hasil :', len(np.unique(depth_image)))

tampil(original, depth_image, 'Grayscale 8-bit', 'Grayscale ' + str(bit_depth) + '-bit')

In [ ]:
# coba semua kedalaman 1 sampai 7 bit
plt.figure(figsize=(16,7))
plt.subplot(2,4,1)
plt.imshow(original, cmap='gray', vmin=0, vmax=255)
plt.title('asli 8bit (256 macam)')
plt.axis('off')
for i, bd in enumerate(range(1,8)):
    lv = 255 / (pow(2, bd) - 1)
    hasil = (np.round(original.astype(float) / lv) * lv).astype(np.uint8)
    plt.subplot(2,4,i+2)
    plt.imshow(hasil, cmap='gray', vmin=0, vmax=255)
    plt.title(str(bd) + 'bit, 0 sampai ' + str(pow(2,bd)-1) + ' (' + str(pow(2,bd)) + ' macam)')
    plt.axis('off')
plt.show()

## 8. Average Denoising

Rata-rata pixel yang koordinatnya sama dari banyak citra ber-noise. Noise gaussian itu acak dan
rata-ratanya nol, jadi kalau dirata-rata noise-nya saling menghilangkan sedangkan gambarnya tetap.

In [ ]:
def PSNR(img1, img2):
    mse = np.mean((img1.astype(float) - img2.astype(float)) ** 2)
    if mse == 0:  # MSE 0 maka tidak ada noise sama sekali, sehingga PSNR tidak memiliki arti
        return 100
    max_pixel = 255.0
    psnr = 20 * math.log10(max_pixel / math.sqrt(mse))
    return psnr

In [ ]:
galaxy = cv.imread(folder + 'galaxy.jpg')

cv_img = []
for img in glob.glob(folder + 'noises/*.jpg'):
    n = cv.imread(img)
    cv_img.append(n)

print('jumlah citra ber-noise :', len(cv_img))
print('PSNR 1 citra ber-noise : %.2f dB' % PSNR(galaxy, cv_img[0]))

plt.figure(figsize=(15,6))
for i in range(10):
    plt.subplot(2,5,i+1)
    plt.imshow(cv.cvtColor(cv_img[i], cv.COLOR_BGR2RGB))
    plt.axis('off')
plt.show()

In [ ]:
def average(daftar, jumlah):
    total = np.zeros(daftar[0].shape, float)
    for im in daftar[:jumlah]:
        total = total + im.astype(float)
    return np.clip(total / jumlah, 0, 255).astype(np.uint8)

hasil = {}
for n in [1, 5, 10, 20, 40, 80, 100]:
    hasil[n] = average(cv_img, n)

plt.figure(figsize=(16,7.5))
plt.subplot(2,4,1)
plt.imshow(cv.cvtColor(galaxy, cv.COLOR_BGR2RGB))
plt.title('citra asli')
plt.axis('off')
for i, n in enumerate([1, 5, 10, 20, 40, 80, 100]):
    plt.subplot(2,4,i+2)
    plt.imshow(cv.cvtColor(hasil[n], cv.COLOR_BGR2RGB))
    plt.title('average ' + str(n) + ' citra')
    plt.axis('off')
plt.show()

### Tabel hasil PSNR

In [ ]:
print('No | Jumlah Citra | PSNR (dB) | Kenaikan dari baris sebelumnya')
sebelum = None
for i, n in enumerate([10, 20, 40, 80, 100]):
    nilai = PSNR(galaxy, hasil[n])
    if sebelum is None:
        naik = '-'
    else:
        naik = '%.2f dB' % (nilai - sebelum)
    print('%d. | %11d | %9.2f | %s' % (i+1, n, nilai, naik))
    sebelum = nilai

In [ ]:
jumlah = [1, 5, 10, 20, 40, 80, 100]
plt.figure(figsize=(7,4))
plt.plot(jumlah, [PSNR(galaxy, hasil[n]) for n in jumlah], 'o-')
plt.xlabel('jumlah citra yang dirata-rata')
plt.ylabel('PSNR (dB)')
plt.grid(alpha=0.3)
plt.show()

## 9. Image Masking

Mask-nya dua lingkaran di posisi wajah, terus citra asli di-AND sama mask itu.

In [ ]:
couple = cv.imread(folder + 'couple.tiff')
h, w = couple.shape[:2]

mask = np.zeros((h, w), np.uint8)
# koordinat lingkaran disesuaikan sama posisi wajah di couple.tiff
kiri = (int(0.30*w), int(0.32*h))
kanan = (int(0.70*w), int(0.32*h))
radius = int(0.16*w)
cv.circle(mask, kiri, radius, 255, -1)
cv.circle(mask, kanan, radius, 255, -1)
mask3 = cv.cvtColor(mask, cv.COLOR_GRAY2BGR)

hasil_and = cv.bitwise_and(couple, mask3)

wajah_kiri = hasil_and[kiri[1]-radius:kiri[1]+radius, kiri[0]-radius:kiri[0]+radius]
wajah_kanan = hasil_and[kanan[1]-radius:kanan[1]+radius, kanan[0]-radius:kanan[0]+radius]

plt.figure(figsize=(17,4))
plt.subplot(1,5,1); plt.imshow(cv.cvtColor(couple, cv.COLOR_BGR2RGB)); plt.title('couple.tiff'); plt.axis('off')
plt.subplot(1,5,2); plt.imshow(mask, cmap='gray'); plt.title('mask'); plt.axis('off')
plt.subplot(1,5,3); plt.imshow(cv.cvtColor(hasil_and, cv.COLOR_BGR2RGB)); plt.title('hasil AND'); plt.axis('off')
plt.subplot(1,5,4); plt.imshow(cv.cvtColor(wajah_kiri, cv.COLOR_BGR2RGB)); plt.axis('off')
plt.subplot(1,5,5); plt.imshow(cv.cvtColor(wajah_kanan, cv.COLOR_BGR2RGB)); plt.axis('off')
plt.show()

Sekarang dicoba operator yang lain. Biar kelihatan jelas bedanya, saya coba dulu di dua region
kotak A dan B seperti gambar di ulasan teori, baru setelah itu di citra couple.

In [ ]:
A = np.zeros((240,340), np.uint8)
B = np.zeros((240,340), np.uint8)
cv.rectangle(A, (30,55), (200,185), 255, -1)
cv.rectangle(B, (150,30), (300,140), 255, -1)

operasi = [('A', A),
           ('B', B),
           ('NOT(A)', cv.bitwise_not(A)),
           ('A OR B', cv.bitwise_or(A,B)),
           ('A AND B', cv.bitwise_and(A,B)),
           ('NAND', cv.bitwise_not(cv.bitwise_and(A,B))),
           ('A XOR B', cv.bitwise_xor(A,B)),
           ('A AND NOT(B)', cv.bitwise_and(A, cv.bitwise_not(B)))]

plt.figure(figsize=(16,7))
for i, (nama, im) in enumerate(operasi):
    plt.subplot(2,4,i+1)
    plt.imshow(im, cmap='gray')
    plt.title(nama)
    plt.axis('off')
plt.show()

In [ ]:
# operator yang sama, tapi input citra couple dan mask lingkaran tadi
uji = [('NOT (komplemen)', cv.bitwise_not(couple)),
       ('OR (atau)', cv.bitwise_or(couple, mask3)),
       ('AND (dan)', cv.bitwise_and(couple, mask3)),
       ('NAND (not and)', cv.bitwise_not(cv.bitwise_and(couple, mask3))),
       ('XOR (exclusive or)', cv.bitwise_xor(couple, mask3))]

plt.figure(figsize=(13,15))
for i, (nama, keluaran) in enumerate(uji):
    plt.subplot(5,2,2*i+1)
    plt.imshow(cv.cvtColor(couple, cv.COLOR_BGR2RGB))
    plt.title('input: couple + mask')
    plt.axis('off')
    plt.subplot(5,2,2*i+2)
    plt.imshow(cv.cvtColor(keluaran, cv.COLOR_BGR2RGB))
    plt.title('output ' + nama)
    plt.axis('off')
plt.tight_layout()
plt.show()

## 10. Foto Malam Hari

Fotonya saya ambil sendiri malam hari, gelap tapi masih ada detail di bagian bangunan. Sebelum
milih, keempat metode saya coba dulu semua.

In [ ]:
malam = cv.imread(folder + 'malam.jpg')   # ganti sama foto malam punya sendiri

coba = [('linear brightness b=70', np.clip(malam.astype(float) + 70, 0, 255).astype(np.uint8)),
        ('linear brightness b=140', np.clip(malam.astype(float) + 140, 0, 255).astype(np.uint8)),
        ('contrast C=70', contrast(malam, 70)),
        ('log c=45', np.clip(45 * np.log(1 + malam.astype(float)), 0, 255).astype(np.uint8)),
        ('gamma correction 3', gamma_correction(malam, 3.0))]

plt.figure(figsize=(14,7))
plt.subplot(2,3,1)
plt.imshow(cv.cvtColor(malam, cv.COLOR_BGR2RGB))
plt.title('foto asli')
plt.axis('off')
for i, (nama, hasil_coba) in enumerate(coba):
    plt.subplot(2,3,i+2)
    plt.imshow(cv.cvtColor(hasil_coba, cv.COLOR_BGR2RGB))
    plt.title(nama, fontsize=9)
    plt.axis('off')
plt.show()

In [ ]:
# bandingkan kecerahan, kontras, dan berapa banyak pixel yang mentok putih (>=250)
abu = cv.cvtColor(malam, cv.COLOR_BGR2GRAY)
print('%-24s: mean %6.1f , std %5.1f , mentok %.2f%%' % ('foto asli', abu.mean(), abu.std(), 100*(abu>=250).mean()))
for nama, hasil_coba in coba:
    a = cv.cvtColor(hasil_coba, cv.COLOR_BGR2GRAY)
    print('%-24s: mean %6.1f , std %5.1f , mentok %.2f%%' % (nama, a.mean(), a.std(), 100*(a>=250).mean()))

## 11. Perbaikan Kualitas crayfish.jpg

Masalah di citra ini ada tiga sekaligus: kontrasnya rendah karena air keruh, warnanya kehijauan
karena cahaya merah paling cepat diserap air, dan pencahayaannya tidak rata. Jadi saya pakai
lebih dari satu metode.

In [ ]:
crayfish = cv.imread(folder + 'crayfish.jpg')
b, g, r = cv.split(crayfish)
abu = cv.cvtColor(crayfish, cv.COLOR_BGR2GRAY)

print('rata-rata channel  -> B: %.1f  G: %.1f  R: %.1f' % (b.mean(), g.mean(), r.mean()))
print('std grayscale      : %.2f' % abu.std())
print('rentang intensitas : %d sampai %d' % (abu.min(), abu.max()))

plt.figure(figsize=(7,3.6))
plt.hist(b.ravel(), bins=64, range=(0,255), histtype='step', label='B')
plt.hist(g.ravel(), bins=64, range=(0,255), histtype='step', label='G')
plt.hist(r.ravel(), bins=64, range=(0,255), histtype='step', label='R')
plt.legend()
plt.title('histogram crayfish.jpg')
plt.show()

In [ ]:
# 1. koreksi warna, samakan rata-rata tiap channel (asumsi gray world)
wb = crayfish.astype(float)
rata = [wb[:,:,i].mean() for i in range(3)]
target = np.mean(rata)
for i in range(3):
    wb[:,:,i] = wb[:,:,i] * target / rata[i]
wb = np.clip(wb, 0, 255).astype(np.uint8)

# 2. CLAHE di channel L
def clahe_lab(img, clip, tile):
    lab = cv.cvtColor(img, cv.COLOR_BGR2LAB)
    L, a, bb = cv.split(lab)
    L = cv.createCLAHE(clipLimit=clip, tileGridSize=(tile,tile)).apply(L)
    return cv.cvtColor(cv.merge([L,a,bb]), cv.COLOR_LAB2BGR)

# coba beberapa clipLimit dulu buat nentuin yang paling pas
plt.figure(figsize=(16,4))
for i, clip in enumerate([1.0, 2.0, 3.0, 5.0]):
    h = clahe_lab(wb, clip, 8)
    plt.subplot(1,4,i+1)
    plt.imshow(cv.cvtColor(h, cv.COLOR_BGR2RGB))
    plt.title('clipLimit = %.1f (std %.1f)' % (clip, cv.cvtColor(h, cv.COLOR_BGR2GRAY).std()))
    plt.axis('off')
plt.show()

In [ ]:
hasil_clahe = clahe_lab(wb, 3.0, 8)
crayfish_final = gamma_correction(hasil_clahe, 1.2)

plt.figure(figsize=(17,4))
for i, (im, judul) in enumerate([(crayfish, '1. asli'), (wb, '2. + white balance'),
                                 (hasil_clahe, '3. + CLAHE'), (crayfish_final, '4. + gamma 1.2')]):
    plt.subplot(1,4,i+1)
    plt.imshow(cv.cvtColor(im, cv.COLOR_BGR2RGB))
    plt.title(judul)
    plt.axis('off')
plt.show()

tampil(crayfish, crayfish_final, 'BEFORE', 'AFTER')

In [ ]:
for nama, im in [('before', crayfish), ('after', crayfish_final)]:
    a = cv.cvtColor(im, cv.COLOR_BGR2GRAY)
    print('%-7s: mean %.2f , std %.2f , rentang %d-%d , laplacian var %.1f'
          % (nama, a.mean(), a.std(), a.min(), a.max(), cv.Laplacian(a, cv.CV_64F).var()))

plt.figure(figsize=(12,3.6))
for i, (im, judul) in enumerate([(crayfish,'histogram before'), (crayfish_final,'histogram after')]):
    plt.subplot(1,2,i+1)
    for c, nama in zip(range(3), ['B','G','R']):
        plt.hist(im[:,:,c].ravel(), bins=64, range=(0,255), histtype='step', label=nama)
    plt.title(judul)
    plt.legend(fontsize=8)
plt.show()